## Датасет собран из базы данных переписи 1994 года и содержит данные о доходах.
### Информация о данных:
* age: continuous.
* workclass: Private, Self-emp-not-inc, Self-emp-inc, Federal-gov, Local-gov, State-gov, Without-pay, Never-worked.
* fnlwgt: continuous.
* education: Bachelors, Some-college, 11th, HS-grad, Prof-school, Assoc-acdm, Assoc-voc, 9th, 7th-8th, 12th, * Masters, 1st-4th, 10th, Doctorate, 5th-6th, Preschool.
* education-num: continuous.
* marital-status: Married-civ-spouse, Divorced, Never-married, Separated, Widowed, Married-spouse-absent, Married-AF-spouse.
* occupation: Tech-support, Craft-repair, Other-service, Sales, Exec-managerial, Prof-specialty, Handlers-cleaners, Machine-op-inspct, Adm-clerical, Farming-fishing, Transport-moving, Priv-house-serv, Protective-serv, Armed-Forces.
* relationship: Wife, Own-child, Husband, Not-in-family, Other-relative, Unmarried.
* race: White, Asian-Pac-Islander, Amer-Indian-Eskimo, Other, Black.
* sex: Female, Male.
* capital-gain: continuous.
* capital-loss: continuous.
* hours-per-week: continuous.
* native-country: United-States, Cambodia, England, Puerto-Rico, Canada, Germany, Outlying-US(Guam-USVI-etc), India, Japan, Greece, South, China, Cuba, Iran, Honduras, Philippines, Italy, Poland, Jamaica, Vietnam, Mexico, Portugal, Ireland, France, Dominican-Republic, Laos, Ecuador, Taiwan, Haiti, Columbia, Hungary, Guatemala, Nicaragua, Scotland, Thailand, Yugoslavia, El-Salvador, Trinadad&Tobago, Peru, Hong, Holand-Netherlands.
* salary: >50K,<=50K

## Проведите анализ данных при помощи Pandas выполнив поставленные задачи.
#### 

In [1]:
import pandas as pd

In [2]:
# загружаем датасет
data = pd.read_csv("./data/adult.data.csv")
data.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


**1. Посчитайте, сколько мужчин и женщин (признак *sex*) представлено в этом датасете**

In [3]:

data["sex"].value_counts()

Male      21790
Female    10771
Name: sex, dtype: int64

**2. Каков средний возраст мужчин (признак *age*) по всему датасету?**

In [4]:
data.loc[data["sex"] == "Male", "age"].mean().round(2)

39.43

**3. Какова доля граждан Соединенных Штатов (признак *native-country*)?**

In [5]:
(data["native-country"] == "United-States").mean() * 100

89.5857006848684

**4-5. Рассчитайте среднее значение и среднеквадратичное отклонение возраста тех, кто получает более 50K в год (признак *salary*) и тех, кто получает менее 50K в год**

In [6]:
data.groupby("salary")["age"].agg(["mean", "std"]).round(2)

,mean,std
salary,,
<=50K,36.78,14.02
>50K,44.25,10.52


**6. Правда ли, что люди, которые получают больше 50k, имеют минимум высшее образование? (признак *education – Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters* или *Doctorate*)**

In [7]:
HIGHER = ["Bachelors", "Prof-school", "Assoc-acdm", "Assoc-voc", "Masters", "Doctorate"]
rich = data[data["salary"] == ">50K"]

print(rich["education"].isin(HIGHER).all())
print((~rich["education"].isin(HIGHER)).sum(), "of", len(rich))
rich.loc[~rich["education"].isin(HIGHER), "education"].value_counts().head()

False
3306 of 7841


HS-grad         1675
Some-college    1387
10th              62
11th              60
7th-8th           40
Name: education, dtype: int64

**7. Выведите статистику возраста для каждой расы (признак *race*) и каждого пола. Используйте *groupby* и *describe*. Найдите таким образом максимальный возраст мужчин расы *Asian-Pac-Islander*.**

In [8]:
stats = data.groupby(["race", "sex"])["age"].describe()
stats.loc[("Asian-Pac-Islander", "Male"), "max"]

90.0

**8. Среди кого больше доля зарабатывающих много (>50K): среди женатых или холостых мужчин (признак *marital-status*)? Женатыми считаем тех, у кого *marital-status* начинается с *Married* (Married-civ-spouse, Married-spouse-absent или Married-AF-spouse), остальных считаем холостыми.**

In [9]:
men = data[data["sex"] == "Male"].copy()
men["married"] = men["marital-status"].str.startswith("Married")

men.groupby("married")["salary"].apply(lambda s: (s == ">50K").mean() * 100).round(2)

married
False     8.45
True     44.05
Name: salary, dtype: float64

**9. Какое максимальное число часов человек работает в неделю (признак *hours-per-week*)? Сколько людей работают такое количество часов и каков среди них процент зарабатывающих много?**

In [10]:
mx = data["hours-per-week"].max()
sub = data[data["hours-per-week"] == mx]

print(mx, len(sub), (sub["salary"] == ">50K").mean() * 100)

99 85 29.411764705882355


**10. Посчитайте среднее время работы (*hours-per-week*) зарабатывающих мало и много (*salary*) для каждой страны (*native-country*).**

In [11]:
data.pivot_table(index="native-country", columns="salary",
                 values="hours-per-week", aggfunc="mean").round(2)

salary,<=50K,>50K
native-country,,
?,40.16,45.55
Cambodia,41.42,40.00
Canada,37.91,45.64
China,37.38,38.90
Columbia,38.68,50.00
Cuba,37.99,42.44
Dominican-Republic,42.34,47.00
Ecuador,38.04,48.75
El-Salvador,36.03,45.00


**11.Сгруппируйте людей по возрастным группам *young*, *adult*, *retiree*, где:**
* *young* соответствует 16-35 лет
* *adult* - 35-70 лет
* *retiree* - 70-100 лет

**Проставьте название соответсвтуещей группы для каждого человека в новой колонке AgeGroup**

In [12]:
data["AgeGroup"] = pd.cut(data["age"],
                          bins=[16, 35, 70, 100],
                          labels=["young", "adult", "retiree"],
                          include_lowest=True)

data["AgeGroup"].value_counts().sort_index()

young      14925
adult      17096
retiree      540
Name: AgeGroup, dtype: int64

**12-13. Определите количество зарабатывающих >50K в каждой из возрастных групп (колонка AgeGroup), а также выведите название возрастной группы, в которой чаще зарабатывают больше 50К (>50K)**

In [13]:
data[data["salary"] == ">50K"]["AgeGroup"].value_counts()

data.groupby("AgeGroup", observed=True)["salary"] \
    .apply(lambda s: (s == ">50K").mean() * 100).round(2)

AgeGroup
young      11.42
adult      35.34
retiree    17.41
Name: salary, dtype: float64

**14. Сгруппируйте людей по типу занятости (колонка occupation) и определите количество людей в каждой группе. После чего напишите функциюю фильтрации filter_func, которая будет возвращать только те группы, в которых средний возраст (колонка age) не больше 40 и в которых все работники отрабатывают более 5 часов в неделю (колонка hours-per-week)**

In [14]:
grouped = data.groupby("occupation")
grouped.size().sort_values(ascending=False)



occupation
Prof-specialty       4140
Craft-repair         4099
Exec-managerial      4066
Adm-clerical         3770
Sales                3650
Other-service        3295
Machine-op-inspct    2002
?                    1843
Transport-moving     1597
Handlers-cleaners    1370
Farming-fishing       994
Tech-support          928
Protective-serv       649
Priv-house-serv       149
Armed-Forces            9
dtype: int64

In [15]:
def filter_func(x):
    return x["age"].mean() <= 40 and (x["hours-per-week"] > 5).all()

result = grouped.filter(filter_func)
result["occupation"].unique()

array(['Armed-Forces'], dtype=object)